**Required libraries**

In [1]:
import torch
import torch.nn as nn
import numpy as np
import tiktoken
import urllib.request

print("PyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PyTorch version: 2.14.1+cpu
Device: cpu


**GPT configuration**

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

**MultiHeadAttention**

In [3]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False
    ):
        super().__init__()

        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):

        batch_size, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        )

        keys = keys.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        )

        values = values.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        )

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(
            mask_bool,
            -torch.inf
        )

        attn_weights = torch.softmax(
            attn_scores / (self.head_dim ** 0.5),
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(
            batch_size, num_tokens, self.d_out
        )

        context_vec = self.out_proj(context_vec)

        return context_vec

**LayerNorm**

In [4]:
class LayerNorm(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):

        mean = x.mean(dim=-1, keepdim=True)

        var = x.var(dim=-1, keepdim=True, unbiased=False)

        norm_x = (x - mean) / torch.sqrt(var + self.eps)

        return self.scale * norm_x + self.shift

**GELU activation**

In [5]:
class GELU(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, x):

        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2.0 / torch.pi))
                * (x + 0.044715 * torch.pow(x, 3))
            )
        )

**FeedForward**

In [6]:
class FeedForward(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):

        return self.layers(x)

**TransformerBlock**

In [7]:
class TransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )

        self.ff = FeedForward(cfg)

        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])

        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):

        shortcut = x

        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)

        x = x + shortcut

        shortcut = x

        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)

        x = x + shortcut

        return x

**GPTModel**

In [8]:
class GPTModel(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(
            cfg["vocab_size"],
            cfg["emb_dim"]
        )

        self.pos_emb = nn.Embedding(
            cfg["context_length"],
            cfg["emb_dim"]
        )

        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[
                TransformerBlock(cfg)
                for _ in range(cfg["n_layers"])
            ]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False
        )

    def forward(self, in_idx):

        batch_size, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)

        pos_indices = torch.arange(
            seq_len,
            device=in_idx.device
        )

        pos_embeds = self.pos_emb(pos_indices)

        x = tok_embeds + pos_embeds

        x = self.drop_emb(x)

        x = self.trf_blocks(x)

        x = self.final_norm(x)

        logits = self.out_head(x)

        return logits

**Tokenizer and text conversion functions**

In [9]:
tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text, tokenizer):

    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

    encoded_tensor = torch.tensor(encoded).unsqueeze(0)

    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):

    flat = token_ids.squeeze(0)

    return tokenizer.decode(flat.tolist())

**Text generation function**

In [10]:
def generate(
    model,
    idx,
    max_new_tokens,
    context_size,
    temperature=0.0,
    top_k=None
):

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -context_size:]

        with torch.no_grad():

            logits = model(idx_cond)

        logits = logits[:, -1, :]

        if top_k is not None:

            top_logits, _ = torch.topk(logits, top_k)

            min_val = top_logits[:, -1]

            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        if temperature > 0.0:

            logits = logits / temperature

            probs = torch.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

        else:

            idx_next = torch.argmax(
                logits,
                dim=-1,
                keepdim=True
            )

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

**Architecture quick test**

In [11]:
test_model = GPTModel(GPT_CONFIG_124M)

test_input = torch.tensor([
    [1, 2, 3, 4]
])

test_output = test_model(test_input)

print("Input shape:", test_input.shape)
print("Output shape:", test_output.shape)

Input shape: torch.Size([1, 4])
Output shape: torch.Size([1, 4, 50257])


**OpenAI original GPT-2 pretrained weigth**

In [12]:
!pip install "tensorflow>=2.15.0" "tqdm>=4.66"

**Download gpt_download.py**

In [13]:
import urllib.request

url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch05/"
    "01_main-chapter-code/gpt_download.py"
)

filename = url.split("/")[-1]

urllib.request.urlretrieve(url, filename)

print("Downloaded:", filename)

Downloaded: gpt_download.py


In [14]:
from gpt_download import download_and_load_gpt2

**GPT-2 Small (124M) download**

In [15]:
settings, params = download_and_load_gpt2(
    model_size="124M",
    models_dir="gpt2"
)

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 270kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:01<00:00, 580kiB/s]
hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 313kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [06:35<00:00, 1.26MiB/s]
model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 13.7MiB/s]
model.ckpt.meta: 100%|██████████| 471k/471k [00:01<00:00, 378kiB/s]
vocab.bpe: 100%|██████████| 456k/456k [00:01<00:00, 380kiB/s]


**Settings aur parameter keys**

In [16]:
print("Settings:")
print(settings)

print("\nParameter dictionary keys:")
print(params.keys())

print("Token embedding shape:", params["wte"].shape)

Settings:
{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}

Parameter dictionary keys:
dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])
Token embedding shape: (50257, 768)


**GPT-2 different model configurations**

In [17]:
model_configs = {
    "gpt2-small (124M)": {
        "emb_dim": 768,
        "n_layers": 12,
        "n_heads": 12
    },

    "gpt2-medium (355M)": {
        "emb_dim": 1024,
        "n_layers": 24,
        "n_heads": 16
    },

    "gpt2-large (774M)": {
        "emb_dim": 1280,
        "n_layers": 36,
        "n_heads": 20
    },

    "gpt2-xl (1558M)": {
        "emb_dim": 1600,
        "n_layers": 48,
        "n_heads": 25
    }
}

**Set GPT-2 Small configuration**

In [18]:
model_name = "gpt2-small (124M)"

NEW_CONFIG = GPT_CONFIG_124M.copy()

NEW_CONFIG.update(model_configs[model_name])

NEW_CONFIG.update({
    "context_length": 1024
})

NEW_CONFIG.update({
    "qkv_bias": True
})

print(NEW_CONFIG)

{'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': True}


**GPT-2 architecture initialize**

In [19]:
gpt = GPTModel(NEW_CONFIG)

gpt.eval()

print("GPT-2 architecture created successfully!")

GPT-2 architecture created successfully!


**Load Open AI Weights**

**assign() function**

In [20]:
def assign(left, right):

    if left.shape != right.shape:

        raise ValueError(
            f"Shape mismatch. Left: {left.shape}, Right: {right.shape}"
        )

    return torch.nn.Parameter(torch.tensor(right))

**OpenAI weights mapping function**

In [21]:
def load_weights_into_gpt(gpt, params):

    # Token aur positional embeddings
    gpt.pos_emb.weight = assign(
        gpt.pos_emb.weight,
        params["wpe"]
    )

    gpt.tok_emb.weight = assign(
        gpt.tok_emb.weight,
        params["wte"]
    )

    # Har transformer block ke weights load karna
    for b in range(len(params["blocks"])):

        # Query, Key, Value weights ko alag karna
        q_w, k_w, v_w = np.split(
            params["blocks"][b]["attn"]["c_attn"]["w"],
            3,
            axis=-1
        )

        # Query, Key, Value weights assign karna
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight,
            q_w.T
        )

        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight,
            k_w.T
        )

        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight,
            v_w.T
        )

        # Query, Key, Value biases alag karna
        q_b, k_b, v_b = np.split(
            params["blocks"][b]["attn"]["c_attn"]["b"],
            3,
            axis=-1
        )

        # QKV biases assign karna
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias,
            q_b
        )

        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias,
            k_b
        )

        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias,
            v_b
        )

        # Attention output projection
        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T
        )

        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"]
        )

        # FeedForward ki first Linear layer
        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T
        )

        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"]
        )

        # FeedForward ki second Linear layer
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T
        )

        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"]
        )

        # First LayerNorm
        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"]
        )

        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"]
        )

        # Second LayerNorm
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"]
        )

        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"]
        )

    # Final LayerNorm
    gpt.final_norm.scale = assign(
        gpt.final_norm.scale,
        params["g"]
    )

    gpt.final_norm.shift = assign(
        gpt.final_norm.shift,
        params["b"]
    )

    # Output head ke weights
    gpt.out_head.weight = assign(
        gpt.out_head.weight,
        params["wte"]
    )

**Actual weights transfer**

In [22]:
load_weights_into_gpt(gpt, params)

print("OpenAI GPT-2 pretrained weights loaded successfully!")

OpenAI GPT-2 pretrained weights loaded successfully!


In [23]:
gpt.to(device)

gpt.eval()

print("Model device:", next(gpt.parameters()).device)

Model device: cpu


**Text generate from pretrained**

In [24]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print(
    "Output text:\n",
    token_ids_to_text(token_ids.cpu(), tokenizer)
)

Output text:
 Every effort moves you toward finding an ideal new way to practice something!

What makes us want to be on top of that?




**Parameter count check**

In [25]:
total_params = sum(
    p.numel() for p in gpt.parameters()
)

print(f"Total parameters: {total_params:,}")
print(f"Total parameters in millions: {total_params / 1_000_000:.2f}M")

Total parameters: 163,037,184
Total parameters in millions: 163.04M


In [26]:
test_prompt = "Artificial intelligence is"

test_ids = text_to_token_ids(test_prompt, tokenizer).to(device)

generated_ids = generate(
    model=gpt,
    idx=test_ids,
    max_new_tokens=30,
    context_size=NEW_CONFIG["context_length"],
    top_k=40,
    temperature=1.0
)

print(token_ids_to_text(generated_ids.cpu(), tokenizer))

Artificial intelligence is a much richer, more powerful technology with a long past that is now becoming apparent at the intersection of machine learning and robotics.

One of the
